02_eda_ws3d.ipynb (EDA pre WS3D)
Cieľ notebooku
- distribúcia tried,
- distribúcia subjektov (speaker-independent split dáva zmysel),
- tvary features/spektrogramov,
- pár vizualizácií.

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = os.path.abspath("..")  # ak notebook beží v projekt/notebooks
WS3D_PROC = os.path.join(PROJECT_ROOT, "data", "processed", "ws3d")

FEAT_DIR = os.path.join(WS3D_PROC, "features")
SPEC_DIR = os.path.join(WS3D_PROC, "spectrograms")

# rekurzívne - aby našlo aj session__*/ses_*.npy
feat_files = sorted(glob.glob(os.path.join(FEAT_DIR, "**", "*.npy"), recursive=True))
spec_files = sorted(glob.glob(os.path.join(SPEC_DIR, "**", "*.npy"), recursive=True))

len(feat_files), len(spec_files), feat_files[:3]

In [ ]:
WS3D_LABEL_MAPPING = {
    "a":  1,   # angry
    "n":  0,   # neutral
    "h":  0,   # happy
    "sa": 1,   # sadness
    "d":  1,   # disgust
    "f":  1,   # fear
    "ps": 0,   # pleasant surprise
    "c":  0,   # calm
}

def parse_ws3d_prefix(path: str) -> str:
    # ses_a03.npy -> "a03"
    stem = os.path.basename(path).replace(".npy", "")
    parts = stem.split("_")
    if len(parts) < 2:
        raise ValueError(f"Nečakaný názov: {path}")
    code = parts[1]  # a03 / n04 / h07 / sa01 ...
    return code

def code_to_key(code: str) -> str:
    # "sa01" -> "sa", inak "a03"->"a"
    if code.startswith("sa"):
        return "sa"
    if code.startswith("ps"):
        return "ps"
    # fallback: prvý znak
    return code[0]

def ws3d_label(path: str) -> int:
    code = parse_ws3d_prefix(path)
    key = code_to_key(code)
    if key not in WS3D_LABEL_MAPPING:
        raise ValueError(f"Neznámy prefix '{key}' z '{code}' pre súbor {path}")
    return WS3D_LABEL_MAPPING[key]

codes = [parse_ws3d_prefix(f) for f in feat_files]
labels = [ws3d_label(f) for f in feat_files]

pd.Series(codes).value_counts().head(10), pd.Series(labels).value_counts().sort_index()

In [ ]:
def parse_session_dir(path: str) -> str:
    # .../features/session__a03/ses_a03.npy -> session__a03
    return os.path.basename(os.path.dirname(path))

sessions = [parse_session_dir(f) for f in feat_files]
pd.Series(sessions).value_counts().head(15)

In [ ]:
counts = pd.Series(labels).value_counts().sort_index()
plt.figure(figsize=(4,3))
counts.plot(kind="bar")
plt.title("WS3D: counts per label (0=no-stress, 1=stress)")
plt.xlabel("label")
plt.ylabel("count")
plt.grid(axis="y", alpha=0.3)
plt.show()

x = np.load(feat_files[0])
print("Example feature shape:", x.shape)